# 🎯 [MNT04_M02] 고주파 센서 신호 잡음 제거, FFT 변환 및 주요 특징량(RMS, Kurtosis) 전처리 구현
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MNT04/mnt04_m02_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장비정비·수리부속 (Level 3) |
| **미션 ID** | MNT04_M02 |
| **미션 주제** | 고주파 센서 신호 잡음 제거, FFT 변환 및 주요 특징량(RMS, Kurtosis) 전처리 구현 |
| **소요 시간** | 약 2시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
항공·기갑 장비의 엔진과 구동계는 고장이 나기 전에 진동, 온도, 압력에 미세한 변화가 먼저 나타나지만, 고정 임계값만으로는 고부하 운용일을 이상으로 오인하거나 초기 열화를 놓쳐 수리부속을 제때 확보하지 못한다. 본 프로젝트는 고장 실적과 수리부속 소모를 연결해 정비와 부품 관리의 연계 문제를 학습하는 것을 목표로, 교육용 가상 기동체 48대의 센서 시계열과 부품 소모를 코드로 만들어 분석한다. 센서 데이터셋 정의, 잡음 제거와 FFT 특징량 추출, 임계값 Baseline과 부품 소모 파레토 분석, 오토인코더 기반 미세 변화 감지와 파생 피처 모델, 잔존수명(RUL) 예측과 이상 신뢰도 점수, 설명가능한 경보 화면, 검사관 승인(HITL) 기반 조기 입고 시뮬레이터, 대용량 스트림 스트레스 시험, AAR 오경보 분석과 임계 재보정, 재학습을 포함한 통합 도구까지 단계별로 구현한다. 최종 목표는 고장 전에 이상 징후를 일찍 잡아 정비 시점과 수리부속 소요를 예고하되, 조기 입고 결정은 검사관이 근거를 보고 내리도록 지원하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 정비 야전(정비대대, 군수지원사령부)에 즉시 도입 가능한 "완성형 국방 예지정비(PdM) 및 PBL 품목관리 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 도메인 및 데이터 규격서** | 군수 도메인 분석서 및 데이터 딕셔너리(`docs/mnt04_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/mnt04_operations_guide.md`) | 미션 1, 10 |
| **`src/etl/`** | **센서 데이터 정제 및 전처리 모듈** | 고주파 센서 신호 잡음 제거, 결측치 보정 및 FFT 특징 추출 파이프라인(`src/etl/mnt04_data_cleansing.py`) | 미션 2 |
| **`notebooks/`** | **탐색적 데이터 분석(EDA) & Baseline** | 3-시그마 임계값 Baseline 및 부품 소모 파레토 분석(`notebooks/mnt04_eda_analysis.ipynb`) | 미션 3 |
| **`src/features/`** | **전술 피처 엔지니어링** | 오토인코더 기반 미세 진동 변화 감지 및 파생 피처 엔지니어링 모듈(`src/features/mnt04_feature_engineering.py`) | 미션 4 |
| **`src/models/`** | **AI 예측 & 잔존수명(RUL) 엔진** | 부품 RUL 예측 앙상블 모델 및 종합 상태 이상 신뢰도 엔진(`src/models/mnt04_advanced_model.py`) | 미션 5 |
| **`src/explainability/`** | **XAI 설명가능성 모듈** | 오토인코더 재구성 오차 분해 및 XAI 원인 설명기(`src/explainability/mnt04_xai_explainer.py`) | 미션 6 |
| **`src/optimization/`** | **HITL 조기 입고 시뮬레이터** | 검사관 HITL 승인 큐, 감사 추적 및 조기 정비 시뮬레이터(`src/optimization/mnt04_simulator.py`) | 미션 7 |
| **`src/dashboard/`** | **스트림 주입 & 관제 대시보드** | 대용량 센서 스트림 주입 스트레스 시험 및 실시간 관제 대시보드(`src/dashboard/mnt04_app.py`) | 미션 8 |
| **`reports/`** | **검증 및 AAR 리포트** | 사후검토(AAR) 오경보 원인 분석 및 임계 파라미터 재보정 ROI 보고서(`reports/mnt04_roi_analysis.md`) | 미션 9 |
| **`pipeline/`** | **MLOps 자동 재학습** | 신규 데이터 기반 챔피언-도전자 자동 재학습 및 E2E 통합 파이프라인(`pipeline/mnt04_retrain.py`) | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. 고주파 센서 신호 잡음 제거, FFT 변환 및 주요 특징량(RMS, Kurtosis) 전처리 구현
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

센서 잡음 제거, FFT 변환 및 주요 특징량(RMS 등) 추출 전처리 모듈을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **고주파 센서 신호의 잡음을 제거하고 FFT 변환으로 RMS, 첨도 등 주요 특징량을 추출하는 전처리 모듈을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **표기·단위(온도 F/C, 압력 psi/kPa) 통일, 중복 제거, 불가능값 결측 처리, 파형 단발 스파이크의 중앙값 필터 제거, 간섭 잡음 창의 저역통과 필터, KNN 결측 보정 등 정제 규칙 10종을 적용 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/etl/mnt04_data_cleansing.py 및 정제 품질 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 특수 예외(비계획 정비, 긴급 불출 등) 데이터 보존의 타당성

#### 미션 PASS 조건

- 정량 기준: 데이터 정제 규칙 10종 적용, 데이터 정합성 검증 통과율 100%


- 진행 기준: ETL 파이프라인 구현 및 정제된 마스터 데이터셋 구축


- 검증 방법: 전후 데이터 통계량(기술통계) 비교 및 무결성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
# ============================================================================
# [미션 02 산출물 파일 연계]: src/etl/mnt04_data_cleansing.py
# (선택) 로컬 모듈 파일로 추출 저장 시: %%writefile src/etl/mnt04_data_cleansing.py
# ============================================================================
"""표기 불일치·중복·단위 혼재·불가능값·결측·단발 스파이크가 섞인 가상 센서 원천 자료에 정제 규칙 10종(파형 스파이크 제거, 저역통과 잡음 제거, KNN 결측 보정 포함)을 적용하고, FFT 변환으로 RMS·첨도 등 주요 특징량을 추출한다. 고장 임박 시점의 긴급 점검 측정은 보존하며 정합성 검증 통과율과 정제 전후 통계를 확인한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kurtosis
from scipy.signal import medfilt, butter, sosfiltfilt
from sklearn.impute import KNNImputer

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 기동체 48대의 엔진·구동계 진동·온도·압력 센서 자료 생성기 (실제 장비·부대·정비 실적 없음)
# 개체마다 정상 -> 열화 -> 고장(수명종료)까지 하루 1회 측정(진동 파형 256표본 + 온도 + 압력 + 부하)을 만든다.
# 열화 유형 4가지: bearing(베어링), gear(기어), oil_pump(오일 펌프), cooling(냉각 팬 모터)
# ---------------------------------------------------------------------------
FS, NS = 1024, 256                              # 가상 표본화 주파수(Hz), 파형 1개당 표본 수
TT = np.arange(NS) / FS
MODES = {  # 열화 유형: (발생 확률, 평균 수명(일), 교체 수리부속, 가상 단가지수)
    "bearing": (0.40, 160, "주 베어링 세트", 6.0), "gear": (0.25, 175, "구동 기어 세트", 9.0),
    "oil_pump": (0.20, 150, "오일 펌프", 4.0), "cooling": (0.15, 140, "냉각 팬 모터", 3.0)}
KIT_COST = 1.0                                  # 모든 정비에 함께 소모되는 씰·필터 키트 단가지수
BANDS = {"b50": (40, 60), "b100": (90, 110), "b150": (140, 156), "b165": (156, 176), "b200": (190, 215)}


def make_runs(n=48, seed=1):
    """반환: units(개체 정보), snap(일별 측정 요약), W(진동 파형 [측정 수, 256])"""
    rng = np.random.default_rng(seed)
    names = list(MODES)
    mode = rng.choice(names, n, p=[MODES[m][0] for m in names])
    life = np.clip(np.array([rng.normal(MODES[m][1], 18) for m in mode]), 90, 230).astype(int)
    start = np.sort(rng.integers(0, 240, n))
    units = pd.DataFrame({"unit": [f"U{i:02d}" for i in range(n)], "mode": mode, "life": life, "start": start, "base_load": rng.uniform(0.4, 0.75, n)})
    rows = []
    for u in units.itertuples():
        d = np.arange(u.life + 1)
        onset = 0.45 * u.life
        deg = np.clip((d - onset) / (u.life - onset), 0, 1) ** 1.5
        load = np.clip(u.base_load + rng.normal(0, 0.06, len(d)), 0.2, 1.0)
        hi = rng.random(len(d)) < 0.07                                            # 고부하 운용일(정상이지만 진동·온도 상승)
        load = np.where(hi, rng.uniform(0.95, 1.1, len(d)), load)
        rows.append(pd.DataFrame({"unit": u.unit, "day": d, "cal": u.start + d, "rul": u.life - d, "deg": deg, "load": load, "hi_load": hi, "mode": u.mode}))
    snap = pd.concat(rows, ignore_index=True)
    S = len(snap)
    snap["ambient"] = 15 + 12 * np.sin(2 * np.pi * snap["cal"] / 365) + rng.normal(0, 2, S)
    dg, ld, md = snap["deg"].values, snap["load"].values, snap["mode"].values
    isb, isg, iso, isc = (md == "bearing"), (md == "gear"), (md == "oil_pump"), (md == "cooling")
    snap["temp"] = 55 + 22 * ld + 0.5 * snap["ambient"] + 9 * dg * iso + 26 * dg * isc + 3 * dg * (isb | isg) + rng.normal(0, 1.5, S)
    snap["press"] = 450 - 12 * ld - 85 * dg * iso - 15 * dg * isc + rng.normal(0, 6, S)
    ph = rng.uniform(0, 2 * np.pi, (S, 3))
    a1, a2 = 1.0 * (0.5 + ld), 0.4 * (0.5 + ld)
    W = (a1[:, None] * np.sin(2 * np.pi * 50 * TT + ph[:, :1]) + a2[:, None] * np.sin(2 * np.pi * 100 * TT + ph[:, 1:2])
         + rng.normal(0, 0.35, (S, NS)))
    W += (0.9 * dg * isb)[:, None] * np.sin(2 * np.pi * 165 * TT + ph[:, 2:3])
    W += (0.8 * dg * isg)[:, None] * (np.sin(2 * np.pi * 150 * TT + ph[:, 2:3]) + 0.8 * np.sin(2 * np.pi * 100 * TT))
    W += (0.5 * dg * iso)[:, None] * np.sin(2 * np.pi * 200 * TT + ph[:, 2:3])
    k = rng.poisson(10 * dg * isb)                                                 # 베어링 열화: 충격성 펄스(첨도 상승)
    pos, amp = rng.integers(0, NS, (S, 30)), rng.normal(0, 3.0, (S, 30)) * dg[:, None]
    amp = amp * (np.arange(30)[None, :] < k[:, None])
    np.add.at(W, (np.repeat(np.arange(S), 30), pos.ravel()), amp.ravel())
    return units, snap, W


def extract(W):
    """진동 파형 -> 특징량: RMS, 첨도, 파고율, 최대-최소, FFT 대역 에너지(50/100/150/165/200Hz 부근)"""
    x = W - W.mean(axis=1, keepdims=True)
    rms = np.sqrt((x ** 2).mean(axis=1))
    out = {"rms": rms, "kurt": kurtosis(x, axis=1, fisher=False), "crest": np.abs(x).max(axis=1) / rms, "p2p": x.max(axis=1) - x.min(axis=1)}
    A = np.abs(np.fft.rfft(x * np.hanning(NS), axis=1)) / NS
    f = np.fft.rfftfreq(NS, 1 / FS)
    for nm, (lo, hi) in BANDS.items():
        out[nm] = np.sqrt((A[:, (f >= lo) & (f < hi)] ** 2).sum(axis=1))
    return pd.DataFrame(out)

# 1) [교육용가상자료] 원천 자료: 생성 자료에 일부러 오류를 섞는다
units, snap, W0 = make_runs()
rng = np.random.default_rng(2)
S = len(snap)
W = W0.copy()
raw = pd.DataFrame({"unit": snap["unit"].values, "date": [(pd.Timestamp("2026-01-01") + pd.Timedelta(days=int(c))).strftime("%Y-%m-%d") for c in snap["cal"]],
                    "temp": snap["temp"].round(1).values, "temp_unit": "C", "press": snap["press"].round(1).values, "press_unit": "kPa",
                    "load": snap["load"].round(2).values, "emergency": snap["rul"].values <= 3})
idx = rng.choice(S, 1500, replace=False)
a, b, c, d_, e, f, g, h_ = np.split(idx, [150, 350, 600, 750, 800, 850, 1050])
raw.loc[a, "unit"] = rng.choice(["u", " U", "unit-"], len(a)) + raw.loc[a, "unit"].str[1:]                 # ID 표기 불일치
raw.loc[b, "date"] = pd.to_datetime(raw.loc[b, "date"]).dt.strftime("%Y/%m/%d")                              # 날짜 형식 혼재
raw.loc[c, "temp"] = (raw.loc[c, "temp"] * 9 / 5 + 32).round(1); raw.loc[c, "temp_unit"] = "F"             # 온도 단위 혼재
raw.loc[d_, "press"] = (raw.loc[d_, "press"] / 6.895).round(1); raw.loc[d_, "press_unit"] = "psi"          # 압력 단위 혼재
raw.loc[e, "temp"] = -999.0; raw.loc[f, "press"] = -5.0                                                    # 불가능 값
raw.loc[g, ["temp", "press"]] = np.nan                                                                     # 결측
sp = h_[:200]                                                                                              # 파형 단발 스파이크(센서 글리치)
for i in sp:
    W[i, rng.integers(0, NS, 2)] += rng.choice([-9.0, 9.0], 2)
nz = h_[200:]                                                                                              # 전자기 간섭: 파형 전체 잡음 증가
W[nz] += rng.normal(0, 0.9, (len(nz), NS))
raw = pd.concat([raw, raw.sample(80, random_state=1)], ignore_index=True)                                   # 중복 80건
print(f"원천 자료 {len(raw)}행 (정상 {S}행 + 중복 80행, 오류 주입 {len(idx)}행, 파형 스파이크 {len(sp)}개·간섭 {len(nz)}개 포함)")

# 2) 정제 규칙 10종 (표 규칙 R1~R6, 결측 R9, 예외 R10 / 파형 규칙 R7~R8)
df, applied = raw.iloc[:S].copy(), []
def rule(name, n):
    applied.append((name, int(n)))

rule("R1 중복 행 제거", raw.duplicated().sum())
uid = "U" + df["unit"].str.extract(r"(\d+)")[0]; rule("R2 개체 ID 표기 통일", (uid != df["unit"]).sum()); df["unit"] = uid
t1 = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce"); t2 = pd.to_datetime(df["date"], format="%Y/%m/%d", errors="coerce")
rule("R3 측정일 형식 통일", t1.isna().sum()); df["date"] = t1.fillna(t2)
fm = df["temp_unit"] == "F"; df.loc[fm, "temp"] = ((df.loc[fm, "temp"] - 32) * 5 / 9).round(1); df["temp_unit"] = "C"; rule("R4 온도 단위 통일(F -> C)", fm.sum())
pm = df["press_unit"] == "psi"; df.loc[pm, "press"] = (df.loc[pm, "press"] * 6.895).round(1); df["press_unit"] = "kPa"; rule("R5 압력 단위 통일(psi -> kPa)", pm.sum())
bt, bp = ~df["temp"].between(-30, 150) & df["temp"].notna(), (df["press"] < 0) & df["press"].notna()
df.loc[bt, "temp"] = np.nan; df.loc[bp, "press"] = np.nan; rule("R6 불가능 값 결측 처리", bt.sum() + bp.sum())
Wc = W.copy()
glitch = np.abs(Wc - np.median(Wc, axis=1, keepdims=True)).max(axis=1) > 8                               # 단발 스파이크 창
for i in np.where(glitch)[0]:
    Wc[i] = medfilt(Wc[i], 3)
rule("R7 파형 단발 스파이크 제거(중앙값 필터)", glitch.sum())
sos = butter(4, 250, fs=FS, output="sos")
hf = (Wc - sosfiltfilt(sos, Wc, axis=1)).std(axis=1)                                                       # 250Hz 이상 성분의 크기
noisy = hf > 0.4                                                                                          # 간섭 잡음 창
Wc[noisy] = sosfiltfilt(sos, Wc[noisy], axis=1); rule("R8 간섭 잡음 창 저역통과(250Hz) 필터", noisy.sum())
miss = df[["temp", "press"]].isna().sum().sum()
X = df.assign(un=df["unit"].str[1:].astype(int) / 50)[["temp", "press", "load", "un"]]
df[["temp", "press"]] = KNNImputer(n_neighbors=5).fit_transform(X)[:, :2]; rule("R9 센서 결측 KNN 보정", miss)
df["rms"] = np.sqrt(((Wc - Wc.mean(axis=1, keepdims=True)) ** 2).mean(axis=1)); lim = df["rms"].quantile(0.99)
df["outlier_flag"] = (df["rms"] > lim) & ~df["emergency"]; rule("R10 긴급 점검 측정 보존(일반 이상값은 표시만)", (df["rms"] > lim).sum() - df["outlier_flag"].sum())
print("\n[적용 규칙과 처리 건수]")
for nme, n in applied:
    print(f" {nme:36s} {n:5d}건")

# 3) 정합성 검증
checks = {"중복 0건": not df.duplicated(["unit", "date"]).any(), "ID 형식": df["unit"].str.match(r"^U\d{2}$").all(),
          "날짜 변환": df["date"].notna().all(), "온도 단위 C": (df["temp_unit"] == "C").all(), "압력 단위 kPa": (df["press_unit"] == "kPa").all(),
          "온도 범위": df["temp"].between(-30, 150).all(), "압력 0 이상": (df["press"] >= 0).all(), "결측 0건": df[["temp", "press"]].notna().all().all(),
          "파형 스파이크 잔존 0개": not (np.abs(Wc - np.median(Wc, axis=1, keepdims=True)).max(axis=1) > 8).any(),
          f"긴급 점검 {int(df['emergency'].sum())}건 보존": int(df["emergency"].sum()) == int(snap["rul"].le(3).sum())}
rate = np.mean(list(checks.values()))
print(f"\n정합성 검증 {sum(checks.values())}/{len(checks)} 통과, 통과율 {rate:.0%} (목표 100%: {'달성' if rate == 1 else '미달'})")
for k, v in checks.items():
    if not v:
        print(" 실패:", k)

# 4) FFT 특징량: 정제 효과(깨끗한 원본 파형 대비 오차)
def feats(Wx):
    x = Wx - Wx.mean(axis=1, keepdims=True)
    return np.sqrt((x ** 2).mean(axis=1)), kurtosis(x, axis=1, fisher=False)
r0, k0 = feats(W0); r1, k1 = feats(W); r2, k2 = feats(Wc)
bad = glitch | noisy
print("\n[특징량 정제 효과] 스파이크·간섭이 있던 창 " + f"{int(bad.sum())}개의 깨끗한 원본 대비 평균 오차")
print(f" RMS   정제 전 {np.abs(r1 - r0)[bad].mean():.3f} -> 정제 후 {np.abs(r2 - r0)[bad].mean():.3f}")
print(f" 첨도  정제 전 {np.abs(k1 - k0)[bad].mean():.2f} -> 정제 후 {np.abs(k2 - k0)[bad].mean():.2f}")
cmp = pd.DataFrame({"정제 전": [len(raw), pd.to_numeric(raw["temp"], errors="coerce").mean(), raw["temp"].min(), raw["press"].min(), raw["press"].isna().sum()],
                    "정제 후": [len(df), df["temp"].mean(), df["temp"].min(), df["press"].min(), df["press"].isna().sum()]},
                   index=["행 수", "온도 평균", "온도 최소", "압력 최소", "압력 결측"]).round(1)
print("\n[정제 전후 기술통계]"); print(cmp.to_string())

i = int(np.where(glitch)[0][0])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(W[i], label="raw", alpha=0.6); ax[0].plot(Wc[i], label="cleaned"); ax[0].set_title("Waveform with glitch"); ax[0].legend()
f = np.fft.rfftfreq(NS, 1 / FS); ax[1].plot(f, np.abs(np.fft.rfft(Wc[i])) / NS); ax[1].set_title("FFT spectrum (cleaned)"); ax[1].set_xlabel("Hz")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
원천 자료 7813행 (정상 7733행 + 중복 80행, 오류 주입 1500행, 파형 스파이크 200개·간섭 250개 포함)

[적용 규칙과 처리 건수]
 R1 중복 행 제거                              80건
 R2 개체 ID 표기 통일                         150건
 R3 측정일 형식 통일                           200건
 R4 온도 단위 통일(F -> C)                    250건
 R5 압력 단위 통일(psi -> kPa)                150건
 R6 불가능 값 결측 처리                         100건
 R7 파형 단발 스파이크 제거(중앙값 필터)               207건
 R8 간섭 잡음 창 저역통과(250Hz) 필터              347건
 R9 센서 결측 KNN 보정                        500건
 R10 긴급 점검 측정 보존(일반 이상값은 표시만)            11건

정합성 검증 10/10 통과, 통과율 100% (목표 100%: 달성)

[특징량 정제 효과] 스파이크·간섭이 있던 창 554개의 깨끗한 원본 대비 평균 오차
 RMS   정제 전 0.267 -> 정제 후 0.116
 첨도  정제 전 8.31 -> 정제 후 0.86

[정제 전후 기술통계]
         정제 전    정제 후
행 수    7813.0  7733.0
온도 평균    73.2    77.3
온도 최소  -999.0    61.5
압력 최소    -5.0   342.8
압력 결측   201.0     0.0
```
</details>
